# Namespace Isolation Resilience Demo (Offline, API-Key-Free)

This notebook implements the automated namespace-isolation assertion test referenced by
`../09-production-resilience-and-operational-engineering.md`'s bug #2 ("A namespace-scoping bug caught
before it reached live client traffic") and by `../99-Interview-QA.md` Q16 -- an automated test, run
against every retrieval code path, proving that a query scoped to one client's namespace never returns
another client's vectors, **regardless of whether a `namespace` argument is explicitly passed**.

The bug narrative it targets: a debug/admin utility called a retriever helper whose `namespace`
parameter had a default value, and the call site simply didn't pass one -- silently falling back to
whichever namespace the default resolved to, instead of the `client_id` resolved server-side that every
other retrieval path used. This notebook builds that exact shape of helper (first the buggy version,
then the fixed one) and demonstrates the isolation failure and the fix side by side, using the same
`FakePineconeIndex` stand-in from `02_pinecone_vector_search_demo.ipynb`.

In [1]:
import numpy as np
import re

np.random.seed(42)
print("Ready.")

Ready.


## 1. Reuse the fake-embedding function and `FakePineconeIndex`

Identical to `02_pinecone_vector_search_demo.ipynb`, copied here so this notebook runs standalone.

In [2]:
EMBED_DIM = 32

def fake_embed(text: str) -> list:
    """Deterministic bag-of-words hashing embedding -- a stand-in for a real embedding model call."""
    vec = np.zeros(EMBED_DIM)
    words = re.findall(r"[a-zA-Z0-9\-]+", text.lower())
    for w in words:
        idx = hash(w) % EMBED_DIM
        vec[idx] += 1.0
    norm = np.linalg.norm(vec)
    return (vec / norm if norm > 0 else vec).tolist()


class FakePineconeIndex:
    """In-memory stand-in for pinecone.Index -- identical shape to
    02_pinecone_vector_search_demo.ipynb's version."""

    def __init__(self, dimension: int, metric: str = "cosine"):
        self.dimension = dimension
        self.metric = metric
        self._store: dict[str, dict[str, dict]] = {}

    def upsert(self, vectors: list[dict], namespace: str = "") -> dict:
        ns = self._store.setdefault(namespace, {})
        for v in vectors:
            ns[v["id"]] = {"values": v["values"], "metadata": v.get("metadata", {})}
        return {"upserted_count": len(vectors)}

    def _matches_filter(self, metadata: dict, filt: dict | None) -> bool:
        if not filt:
            return True
        for key, cond in filt.items():
            val = metadata.get(key)
            if isinstance(cond, dict):
                if "$eq" in cond and val != cond["$eq"]:
                    return False
                if "$in" in cond and val not in cond["$in"]:
                    return False
            else:
                if val != cond:
                    return False
        return True

    def query(self, vector: list, top_k: int = 5, namespace: str = "",
              filter: dict | None = None, include_metadata: bool = False) -> dict:
        ns = self._store.get(namespace, {})
        q = np.array(vector)
        scored = []
        for doc_id, entry in ns.items():
            if not self._matches_filter(entry["metadata"], filter):
                continue
            v = np.array(entry["values"])
            sim = float(np.dot(q, v) / (np.linalg.norm(q) * np.linalg.norm(v) + 1e-9))
            scored.append((doc_id, sim, entry["metadata"]))
        scored.sort(key=lambda t: t[1], reverse=True)
        matches = [
            {"id": doc_id, "score": score, **({"metadata": md} if include_metadata else {})}
            for doc_id, score, md in scored[:top_k]
        ]
        return {"matches": matches, "namespace": namespace}


index = FakePineconeIndex(dimension=EMBED_DIM, metric="cosine")
print("Fake index created, dimension:", index.dimension)

Fake index created, dimension: 32


## 2. Seed two clients' namespaces, including sensitive-looking staging data

Mirrors the bug narrative's setup: a shared staging environment where both clients' namespaces have
been seeded for QA testing -- exactly the scenario in which a missing-namespace default is dangerous,
because *both* namespaces actually have data sitting behind them, so a wrong default silently "works"
instead of returning nothing.

In [3]:
index.upsert(vectors=[
    {"id": "atlas-status-01",
     "values": fake_embed("Project Atlas Japan localization on track for Q3 launch"),
     "metadata": {"client_id": "eli-lilly", "project_id": "atlas", "doc_type": "status_update"}},
    {"id": "atlas-cost-01",
     "values": fake_embed("SKU-LOC-JP-STD Japanese localization standard package pricing confidential"),
     "metadata": {"client_id": "eli-lilly", "project_id": "atlas", "doc_type": "cost_catalog"}},
], namespace="eli-lilly")

index.upsert(vectors=[
    {"id": "orion-status-01",
     "values": fake_embed("Project Orion EU regulatory submission passed compliance review"),
     "metadata": {"client_id": "astrazeneca", "project_id": "orion", "doc_type": "status_update"}},
], namespace="astrazeneca")

# The "default" namespace a careless helper function's signature might resolve to if no client_id
# is passed -- in the bug narrative, this is whichever namespace happens to be seeded first / most
# recently in the shared staging environment, not an empty, harmless namespace.
DEFAULT_NAMESPACE = "eli-lilly"

print("Seeded eli-lilly:", list(index._store["eli-lilly"].keys()))
print("Seeded astrazeneca:", list(index._store["astrazeneca"].keys()))

Seeded eli-lilly: ['atlas-status-01', 'atlas-cost-01']
Seeded astrazeneca: ['orion-status-01']


## 3. The bug, reproduced: a retriever helper with a defaulted `namespace` parameter

This is the exact shape of the bug in chapter 09's narrative #2: a helper function most call sites use
correctly (always passing `client_id` explicitly), but whose signature *allows* omitting it, silently
falling back to a default. A new debug/admin tool added later, in a hurry, doesn't pass one.

In [4]:
def buggy_retrieve(query_text: str, namespace: str = DEFAULT_NAMESPACE, top_k: int = 5):
    """The pre-fix retriever helper -- namespace has a default, so a forgetful call site compiles and
    runs without error, silently querying whichever namespace the default happens to be."""
    return index.query(vector=fake_embed(query_text), top_k=top_k, namespace=namespace,
                        include_metadata=True)


# The new "show recent retrievals for this session" debug tool -- written for an astrazeneca support
# session, but the call site forgot to pass namespace="astrazeneca".
debug_tool_result = buggy_retrieve("What is the status of Project Atlas?")

print("Debug tool result (namespace NOT passed explicitly):")
for m in debug_tool_result["matches"]:
    print(" ", m["id"], m["metadata"])

# This is the leak: an astrazeneca-facing debug session sees eli-lilly's staging data because the
# helper silently fell back to DEFAULT_NAMESPACE instead of erroring.
leaked = [m for m in debug_tool_result["matches"] if m["metadata"]["client_id"] != "astrazeneca"]
print("\nLeaked cross-tenant matches:", [m["id"] for m in leaked])
assert leaked, "expected the buggy default-namespace helper to leak eli-lilly data in this scenario"
print("Confirmed: omitting namespace silently returned another client's data instead of failing.")

Debug tool result (namespace NOT passed explicitly):
  atlas-cost-01 {'client_id': 'eli-lilly', 'project_id': 'atlas', 'doc_type': 'cost_catalog'}
  atlas-status-01 {'client_id': 'eli-lilly', 'project_id': 'atlas', 'doc_type': 'status_update'}

Leaked cross-tenant matches: ['atlas-cost-01', 'atlas-status-01']
Confirmed: omitting namespace silently returned another client's data instead of failing.


## 4. The fix: `namespace` as a required keyword argument, no default

Chapter 9's fix, verbatim: *"making `namespace` a required keyword argument with no default at the
retriever-helper level (so a missing namespace is a `TypeError` at call time, not a silent
fallback)."*

In [5]:
def fixed_retrieve(query_text: str, *, namespace: str, top_k: int = 5):
    """namespace is keyword-only with NO default -- omitting it is a TypeError at call time,
    not a silent fallback to some other client's data."""
    return index.query(vector=fake_embed(query_text), top_k=top_k, namespace=namespace,
                        include_metadata=True)


# The same careless call site, now against the fixed helper -- fails loudly instead of leaking data.
try:
    fixed_retrieve("What is the status of Project Atlas?")
    raised = False
except TypeError as e:
    raised = True
    print("TypeError raised as expected:", e)

assert raised, "the fixed helper must reject a call with no namespace argument at all"
print("\nConfirmed: a forgotten namespace argument is now a loud failure at call time, not a silent")
print("cross-tenant data leak -- exactly the 'hard fail, never a silent default' discipline chapter 09")
print("names for namespace resolution.")

TypeError raised as expected: fixed_retrieve() missing 1 required keyword-only argument: 'namespace'

Confirmed: a forgotten namespace argument is now a loud failure at call time, not a silent
cross-tenant data leak -- exactly the 'hard fail, never a silent default' discipline chapter 09
names for namespace resolution.


## 5. An automated namespace-isolation assertion test, runnable in CI

This is the test chapter 09 references directly -- *"an automated namespace-isolation assertion test...
run in CI against every retrieval code path, not just the primary ones, so a new debug tool added later
would have failed the same test immediately rather than needing a human to notice in staging."*

The assertion below is deliberately general: for **every** client namespace currently seeded in the
index, and **every** query text drawn from a shared query set, a query scoped to that namespace must
never return a match whose `client_id` metadata differs from the namespace it was queried under. This
is the property that should hold for every retrieval code path in the platform (Chapter 2's primary
retrieval, Chapter 6's recommenders reusing the same index, and any future debug/admin tooling),
regardless of which helper function issued the call.

In [6]:
def assert_namespace_isolation(index: FakePineconeIndex, retrieve_fn, query_texts: list[str]) -> None:
    """Fails loudly (AssertionError) the moment any query scoped to one namespace returns a vector
    tagged with a different client_id -- the exact invariant the chapter 09 bug violated."""
    namespaces = list(index._store.keys())
    violations = []
    for ns in namespaces:
        for q in query_texts:
            result = retrieve_fn(q, namespace=ns, top_k=10)
            for m in result["matches"]:
                leaked_client = m["metadata"]["client_id"]
                if leaked_client != ns:
                    violations.append((ns, q, m["id"], leaked_client))
    assert not violations, f"Namespace isolation violated: {violations}"


query_set = [
    "What is the status of Project Atlas?",
    "What is the regulatory submission status for Project Orion?",
    "What is the pricing for Japanese localization?",
]

# Run the isolation test against the FIXED helper -- this is the "would pass in CI" case.
assert_namespace_isolation(index, fixed_retrieve, query_set)
print("Isolation test PASSED against fixed_retrieve for all namespaces and queries.")

Isolation test PASSED against fixed_retrieve for all namespaces and queries.


In [7]:
# Run the same isolation test against the BUGGY helper's underlying query behavior, but this time
# simulating the debug tool's actual call pattern: a wrapper that silently omits namespace and lets
# buggy_retrieve fall back to DEFAULT_NAMESPACE, regardless of which client's session it's serving.
def buggy_retrieve_as_called_by_debug_tool(query_text: str, namespace: str, top_k: int = 5):
    """Wraps buggy_retrieve the way the real debug tool called it: the caller *believes* it is
    scoping to `namespace` (that's the CI test's contract), but the underlying helper ignores it in
    favor of its own default -- reproducing the exact silent-fallback bug end to end."""
    return buggy_retrieve(query_text, top_k=top_k)  # NOTE: `namespace` argument dropped on the floor


try:
    assert_namespace_isolation(index, buggy_retrieve_as_called_by_debug_tool, query_set)
    print("Isolation test PASSED (unexpected).")
except AssertionError as e:
    print("Isolation test FAILED, as expected for the buggy helper:")
    print(str(e)[:400], "...")

Isolation test FAILED, as expected for the buggy helper:
Namespace isolation violated: [('astrazeneca', 'What is the status of Project Atlas?', 'atlas-cost-01', 'eli-lilly'), ('astrazeneca', 'What is the status of Project Atlas?', 'atlas-status-01', 'eli-lilly'), ('astrazeneca', 'What is the regulatory submission status for Project Orion?', 'atlas-cost-01', 'eli-lilly'), ('astrazeneca', 'What is the regulatory submission status for Project Orion?', 'atl ...


## Takeaways

- **A default `namespace` value is the vulnerability, not the missing call site.** The bug wasn't a
  one-off mistake at a single call site; it was that the helper's signature *permitted* the mistake to
  compile and run silently. Making `namespace` keyword-only with no default converts a silent data leak
  into a `TypeError` at call time (Section 4).
- **The isolation assertion test is generic across retrieval code paths.** `assert_namespace_isolation`
  doesn't care which helper function issued a query -- it re-runs a fixed query set against every
  namespace and asserts no cross-tenant leakage, which is exactly what would catch a *new* debug tool
  added later without anyone touching this test (Section 5).
- **The fix and the test are complementary, not redundant.** The keyword-only signature prevents the
  specific bug class chapter 09 names; the CI assertion test is the backstop that would catch it (or a
  different mistake with the same symptom) even if a future helper reintroduces a default by accident.